# TinyVerse Next-Word Predictor

This executable portfolio notebook follows askThili's *Language Modeling from First Principles*. It builds a word-level n-gram model from counts, without neural networks or third-party packages.

We use the exact 60-sentence course corpus, compare bigrams and trigrams, MLE and add-k smoothing, and report training and held-out perplexity. The Alice in Wonderland experiment is pending because Project Gutenberg returned HTTP 503 during preparation. See DATASETS.md for attribution and copyright notes.

In [1]:
from pathlib import Path
from next_word_predictor import NgramLM, count_ngrams, split_sentences, to_sentences, tokenize
corpus_text = Path("course_corpus.txt").read_text(encoding="utf-8")
sentences = to_sentences(corpus_text)
all_tokens = [token for sentence in sentences for token in tokenize(sentence)]
print(f"Sentences: {len(sentences)} | tokens: {len(all_tokens)} | vocabulary: {len(set(all_tokens))}")
assert len(sentences) == 60

Sentences: 60 | tokens: 493 | vocabulary: 169


## Lesson 1: Tokenization and preprocessing

We lowercase and tokenize with the course pattern [a-z']+. Sentence starts and ends use <s> and </s>. The original corpus text remains unchanged in course_corpus.txt.

In [2]:
print("First sentence tokens:", tokenize(sentences[0]))
bigram_counts = count_ngrams(sentences, n=2)
trigram_counts = count_ngrams(sentences, n=3)
print("After the:", bigram_counts[("the",)].most_common(6))
print("After in the:", trigram_counts[("in", "the")].most_common(5))
print("Sentence starts:", bigram_counts[("<s>",)].most_common(5))

First sentence tokens: ['i', 'drink', 'coffee', 'every', 'morning', 'before', 'work']
After the: [('morning', 7), ('weekend', 6), ('park', 5), ('city', 3), ('train', 3), ('rain', 3)]
After in the: [('park', 4), ('morning', 4), ('evening', 2), ('afternoon', 1), ('rain', 1)]
Sentence starts: [('the', 23), ('i', 8), ('we', 8), ('she', 7), ('he', 7)]


## Lessons 2–3: N-gram counts and maximum likelihood

A bigram uses the previous word as context; a trigram uses the previous two. The count table maps context to following-word counts. Maximum likelihood estimation divides a continuation count by the total count for its context:

P(word | context) = count(context, word) / count(context)

The reusable implementation is in next_word_predictor.py. Its vocabulary is fitted only on training text, and unseen words map to <unk>.

In [3]:
SEED = 42
train_sentences, test_sentences = split_sentences(sentences, test_fraction=0.2, seed=SEED)
train_text = ". ".join(train_sentences) + "."
test_text = ". ".join(test_sentences) + "."
print(f"Train: {len(train_sentences)} sentences | held out: {len(test_sentences)} sentences")
print("Sentence overlap:", len(set(train_sentences) & set(test_sentences)))
assert not (set(train_sentences) & set(test_sentences))
bigram_lm = NgramLM(n=2, k=0.5).fit(train_sentences)
trigram_lm = NgramLM(n=3, k=0.5).fit(train_sentences)
print("Training vocabulary size:", len(bigram_lm.vocabulary))

Train: 48 sentences | held out: 12 sentences
Sentence overlap: 0
Training vocabulary size: 150


## Lessons 3–4: Next-word suggestions and generation

suggest_next ranks likely continuations. Sampling from the distribution one word at a time is autoregressive generation. Fixed seeds make generated examples repeatable; they are illustrations, not evidence of broad language understanding.

In [4]:
for phrase in ["i drink", "the morning", "on the", "she reads a"]:
    print(f"{phrase!r:16} bigram: {bigram_lm.suggest_next(phrase, topk=3)}")
    print(f"{'':16} trigram: {trigram_lm.suggest_next(phrase, topk=3)}")
for n, model in [(2, bigram_lm), (3, trigram_lm)]:
    print(f"n={n}, seed=3: {model.generate(seed=3, max_tokens=25)!r}")
    print(f"n={n}, seed=7: {model.generate(seed=7, max_tokens=25)!r}")

'i drink'        bigram: [('coffee', 0.01948051948051948), ('tea', 0.01948051948051948), ('a', 0.006493506493506494)]
                 trigram: [('coffee', 0.019736842105263157), ('a', 0.006578947368421052), ('after', 0.006578947368421052)]
'the morning'    bigram: [('coffee', 0.02840909090909091), ('train', 0.02840909090909091), ('before', 0.017045454545454548)]
                 trigram: [('train', 0.03125), ('coffee', 0.018750000000000003), ('a', 0.00625)]
'on the'         bigram: [('morning', 0.04263565891472868), ('park', 0.03488372093023256), ('weekend', 0.03488372093023256)]
                 trigram: [('weekend', 0.042682926829268296), ('cold', 0.018292682926829267), ('corner', 0.018292682926829267)]
'she reads a'    bigram: [('good', 0.04069767441860465), ('book', 0.01744186046511628), ('busy', 0.01744186046511628)]
                 trigram: [('book', 0.01948051948051948), ('chapter', 0.01948051948051948), ('a', 0.006493506493506494)]
n=2, seed=3: 'corner morning evenings old ou

## Lessons 5–6: Add-k smoothing, cross-entropy, and perplexity

Add-k smoothing is P(word | context) = (count(context, word) + k) / (count(context) + k × vocabulary_size). We test k = 0 (MLE), 0.1, 0.5, and 1.0. Larger k spreads more probability mass across the training vocabulary.

Cross-entropy is H = -(1/N) sum(log P(w_i | context_i)); perplexity is exp(H). Lower perplexity is better. A single zero-probability MLE event makes perplexity infinite. The held-out split is kept separate from fitting; OOV words map to <unk> using the training vocabulary.

In [5]:
def fmt(value):
    return "infinity" if value == float("inf") else f"{value:.2f}"
rows = []
for n in (2, 3):
    model = NgramLM(n=n, k=0.0).fit(train_sentences)
    for k in (0.0, 0.1, 0.5, 1.0):
        rows.append((n, k, model.perplexity(train_text, k=k), model.perplexity(test_text, k=k)))
print(f"{'n':>2} {'k':>5} {'train PPL':>12} {'held-out PPL':>15}")
for n, k, train_ppl, test_ppl in rows:
    print(f"{n:>2} {k:>5.1f} {fmt(train_ppl):>12} {fmt(test_ppl):>15}")

 n     k    train PPL    held-out PPL
 2   0.0         3.54        infinity
 2   0.1        13.61           65.28
 2   0.5        36.99           80.47
 2   1.0        55.03           92.15
 3   0.0         1.88        infinity
 3   0.1        13.83           93.48
 3   0.5        42.57          107.46
 3   1.0        63.43          116.21


### Reading the results

Training perplexity measures fit on sentences whose counts the model has seen. Held-out perplexity measures prediction on sentences kept outside fitting; it is the more honest estimate of generalization. More context can sound better during generation but creates sparser contexts. These values come from a tiny corpus and one deterministic split, so they are a teaching result rather than a broad benchmark.

In [6]:
mle = NgramLM(n=2, k=0.0).fit(train_sentences)
smoothed = NgramLM(n=2, k=0.5).fit(train_sentences)
context, unseen_word = "drinks", "coffee"
print("MLE P(coffee | drinks):", mle.probabilities(context).get(unseen_word, 0.0))
print("Add-0.5 P(coffee | drinks):", smoothed.probabilities(context).get(unseen_word, 0.0))
print("Empty-input perplexity:", smoothed.perplexity(""))

MLE P(coffee | drinks): 0.0
Add-0.5 P(coffee | drinks): 0.006493506493506494
Empty-input perplexity: nan


## Lesson 7: Reusable NgramLM and larger-corpus status

NgramLM provides fit, probabilities, suggest_next, generate, cross_entropy, and perplexity. It handles empty text, unknown words, unseen contexts, and zero-probability MLE events explicitly.

The Project Gutenberg loader accepts a caller-verified plain-text URL and reads into memory. We do not fetch or report Alice metrics while the source page is unavailable. Verify its current download and terms first; see DATASETS.md.

In [7]:
from next_word_predictor import load_gutenberg_text
GUTENBERG_TEXT_URL = None  # Set only after checking the live ebook page.
if GUTENBERG_TEXT_URL:
    alice_text = load_gutenberg_text(GUTENBERG_TEXT_URL)
    print(f"Fetched {len(alice_text):,} characters into memory; no file was saved.")
else:
    print("Pending: Project Gutenberg ebook 11 returned HTTP 503 when checked.")
    print("No Alice corpus metrics are reported.")

Pending: Project Gutenberg ebook 11 returned HTTP 503 when checked.
No Alice corpus metrics are reported.


## Limitations and running tests

An n-gram model only uses the previous n - 1 tokens. With 60 sentences, trigrams are sparse and generated text may repeat training phrases. Add-k is a simple teaching baseline. Run tests from this folder with python -m unittest discover -s tests -v. No package installation is needed.